# Módulo I2 · Regresión y predicción

**Caso:** un hospital quiere anticipar la **progresión de la diabetes** de sus pacientes a un año vista para priorizar seguimientos. Disponemos de 442 pacientes con 10 variables clínicas (edad, sexo, IMC, presión arterial y seis analíticas de sangre).

Objetivos:
1. Ajustar e **interpretar** una regresión lineal (coeficientes, p-valores, R²).
2. Diagnosticar el modelo con los residuos.
3. Medir el rendimiento **fuera de muestra** con train/test y validación cruzada.
4. Controlar el sobreajuste con regularización (Ridge y Lasso).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")

In [ ]:
data = load_diabetes(as_frame=True, scaled=False)
df = data.frame.rename(columns={
    "age": "edad", "sex": "sexo", "bmi": "imc", "bp": "presion",
    "s1": "colesterol", "s2": "ldl", "s3": "hdl", "s4": "col_hdl", "s5": "trigliceridos", "s6": "glucosa",
    "target": "progresion",
})
df["sexo"] = (df["sexo"] == df["sexo"].max()).astype(int)
df.head()

> **Nota:** en este conjunto de datos, `trigliceridos` (variable original *s5*) está en **escala logarítmica**, y `sexo` vale 1 para uno de los dos sexos y 0 para el otro (el conjunto original no indica cuál). Tenlo en cuenta al interpretar sus coeficientes.

## 1. Exploración

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["progresion"], ax=axes[0]); axes[0].set_title("Variable objetivo")
corr = df.corr()["progresion"].drop("progresion").sort_values()
corr.plot.barh(ax=axes[1]); axes[1].set_title("Correlación con la progresión")
plt.tight_layout(); plt.show()

## 2. Regresión lineal simple: IMC → progresión

In [ ]:
sns.regplot(data=df, x="imc", y="progresion", scatter_kws={"s": 12, "alpha": 0.6})
plt.title("Progresión vs IMC"); plt.show()

simple = sm.OLS(df["progresion"], sm.add_constant(df[["imc"]])).fit()
print(f"Pendiente: {simple.params['imc']:.1f} puntos de progresión por cada punto de IMC")
print(f"R²: {simple.rsquared:.3f}")

## 3. Regresión múltiple con statsmodels
`statsmodels` ofrece la tabla clásica con errores estándar y p-valores: ideal para **explicar**.

In [ ]:
X, y = df.drop(columns="progresion"), df["progresion"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

ols = sm.OLS(y_train, sm.add_constant(X_train)).fit()
print(ols.summary())

**Cómo leer la tabla:**
- `coef`: variación esperada de la progresión al aumentar la variable en 1 unidad, **manteniendo el resto constante**.
- `P>|t|`: si es < 0,05, el coeficiente es estadísticamente distinto de 0.
- Las analíticas están muy correlacionadas entre sí (*multicolinealidad*): sus coeficientes individuales son inestables aunque la predicción conjunta sea buena.

Para comparar la importancia relativa, ajustamos con las variables **estandarizadas**:

In [ ]:
esc = StandardScaler().fit(X_train)
coef_std = pd.Series(LinearRegression().fit(esc.transform(X_train), y_train).coef_, index=X.columns)
coef_std.sort_values().plot.barh(); plt.title("Coeficientes estandarizados"); plt.show()

## 4. Diagnóstico de residuos

In [ ]:
resid, ajust = ols.resid, ols.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(ajust, resid, s=10); axes[0].axhline(0, c="r")
axes[0].set(title="Residuos vs ajustados", xlabel="predicción", ylabel="residuo")
sm.qqplot(resid, line="s", ax=axes[1]); axes[1].set_title("QQ-plot de residuos")
plt.tight_layout(); plt.show()

Buscamos residuos sin patrón y con varianza constante. Un "embudo" indicaría heterocedasticidad; una curva, una relación no lineal no capturada.

## 5. ¿Cómo predice con datos nuevos?

In [ ]:
def evaluar(modelo, nombre):
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    cv = cross_val_score(modelo, X_train, y_train, cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
    return {"modelo": nombre, "R² CV (train)": cv.mean(), "R² test": r2_score(y_test, pred), "MAE test": mean_absolute_error(y_test, pred)}

resultados = [
    evaluar(LinearRegression(), "Lineal"),
    evaluar(make_pipeline(StandardScaler(), PolynomialFeatures(2), LinearRegression()), "Polinómica grado 2"),
    evaluar(make_pipeline(StandardScaler(), PolynomialFeatures(2), RidgeCV(alphas=np.logspace(-2, 4, 50))), "Polinómica + Ridge"),
    evaluar(make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)), "Lasso"),
]
pd.DataFrame(resultados).set_index("modelo").round(3)

La regresión polinómica sin regularizar tiene 65 coeficientes para ~330 pacientes: **sobreajusta** y empeora en test. Ridge "encoge" los coeficientes y recupera el rendimiento. Lasso además pone algunos a **cero exactamente**, lo que equivale a seleccionar variables:

In [ ]:
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)).fit(X_train, y_train)
coefs = pd.Series(lasso[-1].coef_, index=X.columns)
print("Variables descartadas por Lasso:", list(coefs[coefs == 0].index))
coefs.sort_values().plot.barh(); plt.title("Coeficientes Lasso (estandarizados)"); plt.show()

## 6. De la predicción a la decisión
Con un MAE de ~41 puntos, ¿es útil el modelo? Compáralo con la alternativa ingenua de predecir siempre la media:

In [ ]:
print(f"MAE prediciendo la media: {mean_absolute_error(y_test, np.full_like(y_test, y_train.mean())):.1f}")
print(f"MAE del modelo Lasso:     {mean_absolute_error(y_test, lasso.predict(X_test)):.1f}")

## Ejercicios

1. Ajusta un modelo con solo `imc`, `presion` y `trigliceridos`. ¿Cuánto R² de test pierdes frente al modelo completo? ¿Cuál explicarías antes a un médico?
2. Añade la interacción `imc × edad` al modelo de statsmodels (`df["imc_edad"] = df.imc * df.edad`). ¿Es significativa? Interprétala.
3. Representa el error de test de Ridge en función de `alpha` (de 0,01 a 10.000, escala log). Señala la zona de sobreajuste y la de infraajuste.
4. El hospital solo puede hacer seguimiento intensivo al 20 % de pacientes. Usa el modelo para elegirlos en el conjunto de test y calcula qué % del verdadero 20 % con mayor progresión has capturado.

In [ ]:
# TU CÓDIGO AQUÍ